# Savage-Dickey Ratio Demo

Goal: build intuition for testing whether a history-effect coefficient is zero.

For a nested test such as `beta_x = 0`, the Savage-Dickey ratio compares prior and posterior density at the null value:

- `BF01 = posterior_density(0) / prior_density(0)` supports the null.
- `BF10 = prior_density(0) / posterior_density(0)` supports a nonzero effect.

If posterior mass moves away from zero as sample size grows, `posterior_density(0)` shrinks and `BF10` grows.

In [ ]:
from pathlib import Path
import json

import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import gaussian_kde, norm


def find_section_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "section_3" / "src" / "inference.py").exists():
            return path / "section_3"
        if path.name == "section_3" and (path / "src" / "inference.py").exists():
            return path
    raise FileNotFoundError("Could not locate section_3 root.")


section_root = find_section_root(Path.cwd())
latest_path = section_root / "results" / "part_1_complex_latest.txt"
run_rel = latest_path.read_text().strip()
run_dir = section_root / "results" / run_rel
figure_dir = section_root / "figures" / "savage_dickey_demo"
figure_dir.mkdir(parents=True, exist_ok=True)

PARAM_LABELS = {
    "beta_x": r"$\beta_x$",
    "beta_y": r"$\beta_y$",
}

run_dir

## 1. Toy Normal Example

This is not the section 3 model. It is only the simplest picture of the density-ratio idea.

In [ ]:
prior_sd = 1.0
posterior_cases = {
    "null-like posterior": {"mean": 0.02, "sd": 0.18},
    "nonzero posterior": {"mean": 0.80, "sd": 0.18},
}

x = np.linspace(-2.5, 2.5, 800)
prior_pdf = norm.pdf(x, loc=0.0, scale=prior_sd)
prior_at_zero = norm.pdf(0.0, loc=0.0, scale=prior_sd)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), dpi=180, sharey=True)

rows = []
for ax, (label, case) in zip(axes, posterior_cases.items()):
    post_pdf = norm.pdf(x, loc=case["mean"], scale=case["sd"])
    posterior_at_zero = norm.pdf(0.0, loc=case["mean"], scale=case["sd"])
    bf01 = posterior_at_zero / prior_at_zero
    bf10 = prior_at_zero / posterior_at_zero

    ax.plot(x, prior_pdf, color="0.35", linestyle="--", label="prior")
    ax.plot(x, post_pdf, color="#1f77b4", label="posterior")
    ax.axvline(0.0, color="#d62728", linestyle=":", linewidth=1.5, label="null: 0")
    ax.set_title(f"{label}\nBF10={bf10:.2g}")
    ax.set_xlabel(r"$\beta$")
    ax.grid(alpha=0.25)

    rows.append(
        {
            "case": label,
            "prior_density_at_0": prior_at_zero,
            "posterior_density_at_0": posterior_at_zero,
            "BF01": bf01,
            "BF10": bf10,
        }
    )

axes[0].set_ylabel("density")
axes[0].legend(frameon=False)
fig.tight_layout()

toy_bf = pd.DataFrame(rows)
display(toy_bf)

out_path = figure_dir / "toy_savage_dickey_ratio.svg"
fig.savefig(out_path, bbox_inches="tight", transparent=True)
out_path

## 2. Existing Section 3 Fits

The current `operation_1.py` already writes `history_effect_bayes_factors.csv` for `beta_x` and `beta_y` when prior draws are available. This cell gathers those files across cumulative sample sizes.

In [ ]:
bf_rows = []

for n_dir in sorted([p for p in run_dir.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name)):
    bf_path = n_dir / "history_effect_bayes_factors.csv"
    if not bf_path.exists():
        continue

    df = pd.read_csv(bf_path)
    df["n_cell"] = int(n_dir.name)
    bf_rows.append(df)

beta_bf = pd.concat(bf_rows, ignore_index=True) if bf_rows else pd.DataFrame()

if beta_bf.empty:
    print("No history_effect_bayes_factors.csv files found yet.")
else:
    beta_bf["log10_BF10"] = np.log10(beta_bf["BF10"].replace({np.inf: np.nan}))
    display(beta_bf.sort_values(["parameter", "n_cell"]))

In [ ]:
if not beta_bf.empty:
    fig, ax = plt.subplots(figsize=(6, 4), dpi=180)

    for param, sub in beta_bf.sort_values("n_cell").groupby("parameter"):
        y = np.log10(sub["BF10"].replace({np.inf: np.nan}))
        ax.plot(
            sub["n_cell"],
            y,
            marker="o",
            label=PARAM_LABELS.get(param, param),
        )

        inf_mask = np.isinf(sub["BF10"].to_numpy(dtype=float))
        if inf_mask.any():
            finite_y = y[np.isfinite(y)]
            marker_y = finite_y.max() + 1.0 if finite_y.size else 1.0
            ax.scatter(
                sub.loc[inf_mask, "n_cell"],
                np.full(inf_mask.sum(), marker_y),
                marker="^",
                s=45,
            )

    ax.axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_xlabel("Number of in-silico NK cells")
    ax.set_ylabel(r"$\log_{10} BF_{10}$ for nonzero history effect")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()

    out_path = figure_dir / "beta_savage_dickey_bf10_by_cell_number.svg"
    fig.savefig(out_path, bbox_inches="tight", transparent=True)
    display(out_path)